# Report Core

Shared imports, data loading, parameters and core metrics for the Bloomberg billing verification report. Used by `analysis_v1` and investigation notebooks via `%run`.

In [0]:
%run "./utils"

In [0]:
import datetime
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [0]:
robeco_palette = {
    "Blue": "#00A2BD",
    "Orange": "#F79608",
    "Pink": "#B50063",
}

In [0]:
def robeco_bar_chart(data, x=None, y=None, *, title="", xlabel="Month", ylabel="Count", color=None, figsize=(14, 6), stacked=False, legend_title=None, colors=None):
    """Standard Robeco-styled bar chart. For stacked charts, `colors` is an optional list (one per column)
    or dict (column -> colour); without it matplotlib's 10-colour cycle repeats after 10 series."""
    fig, ax = plt.subplots(figsize=figsize)
    if stacked:
        if isinstance(colors, dict):
            colors = [colors[c] for c in data.columns]
        data.plot.bar(stacked=True, ax=ax, width=0.85, color=colors)
    else:
        ax.bar(data[x], data[y], color=color or robeco_palette["Blue"])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=45)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5, color="LightGrey")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)
    if legend_title:
        ax.legend(title=legend_title, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    display(fig)
    plt.close(fig)


def robeco_dumbbell_chart(data, label_col, start_col, end_col, *, title="", start_label="Start", end_label="End", xlabel="Unique Securities", figsize=(12, None), log=False, flag_col=None):
    """Robeco-styled dumbbell chart: one row per label, a dot for start and end values joined by a line.
    log=True uses a log x-axis (products differ by orders of magnitude); flag_col bolds flagged rows."""
    data = data.sort_values(end_col).reset_index(drop=True)
    if log:
        data = data.assign(_s=data[start_col].clip(lower=1), _e=data[end_col].clip(lower=1))
    else:
        data = data.assign(_s=data[start_col], _e=data[end_col])
    height = figsize[1] or max(3, 0.45 * len(data) + 1.5)
    fig, ax = plt.subplots(figsize=(figsize[0], height))
    y = range(len(data))
    ax.hlines(y, data["_s"], data["_e"], color="LightGrey", linewidth=3, zorder=1)
    ax.scatter(data["_s"], y, color=robeco_palette["Blue"], s=60, label=start_label, zorder=2)
    ax.scatter(data["_e"], y, color=robeco_palette["Orange"], s=30, label=end_label, zorder=3)
    for i, row in data.iterrows():
        delta = row[end_col] - row[start_col]
        pct = f" ({delta / row[start_col]:+.1%})" if row[start_col] else " (new)"
        flagged = bool(flag_col and row[flag_col])
        colour = robeco_palette["Pink"] if flagged else "DimGrey"
        ax.annotate(
            f"{row[start_col]:,.0f} -> {row[end_col]:,.0f}   {delta:+,.0f}{pct}",
            (max(row["_s"], row["_e"]), i), xytext=(10, 0), textcoords="offset points",
            va="center", fontsize=8, color=colour, fontweight="bold" if flagged else "normal",
        )
    if log:
        ax.set_xscale("log")
        ax.set_xlim(right=data[["_s", "_e"]].max().max() * 8)
    ax.set_yticks(list(y))
    ax.set_yticklabels(data[label_col], fontsize=9)
    if flag_col:
        for tick, flagged in zip(ax.get_yticklabels(), data[flag_col]):
            if flagged:
                tick.set_color(robeco_palette["Pink"])
                tick.set_fontweight("bold")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.xaxis.grid(True, linestyle="--", alpha=0.5, color="LightGrey")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    display(fig)
    plt.close(fig)

In [0]:
# Account constants — defined before loading so we can filter immediately.
# Account 35922 (GLAC-EGF backfill, Jan-Feb 2024, zero billed usage) is excluded.
MAIN_ACCOUNT = "301491"
PORT_ACCOUNT = "791626"
ACCOUNTS = {MAIN_ACCOUNT: "Main", PORT_ACCOUNT: "PORT"}

ss_df = (
    tables("bbg.verification_sched_summary", "rob6332").current()
    .where(col("ACCOUNT_ID").isin(list(ACCOUNTS)))
    .withColumn("STATUS", F.when(col("TOTAL_USAGE") > 0, "ACTIVE").otherwise("INACTIVE"))
)

sd_df = (
    tables("bbg.verification_sched_detail", "rob6332").current()
    .where(col("ACCOUNT_ID").isin(list(ACCOUNTS)))
    .withColumns(
        {
            "DATE": F.to_date(col("PROCESSED_TIME")),
            "PROCESS_ID": F.when(col("SERVICE") == "F", F.substring(col("OUTPUT_ID"), 1, 3)).otherwise("API"),
            "PROCESS_NAME": process_map[col("PROCESS_ID")],
            "REPORT_YR_MONTH": F.date_format(col("PROCESSED_TIME"), "yyyy/MM"),
            "ENV": F.coalesce(
                F.nullif(F.upper(F.regexp_extract(col("OUTPUT_ID"), r"(?i)_(PROD|PRD|UAT|ACC|TST|TEST|DEV)$", 1)), F.lit("")),
                F.lit("N/A"),
            ),
        }
    )
)

# billed requests only — the default basis for all trend analysis
sd_billed = sd_df.where(col("CTRB_BILLING") == "Y")

In [0]:
# Derive report month from latest data
REPORT_MONTH = (
    ss_df
    .select(F.max("REPORT_YR_MONTH").alias("latest"))
    .first()["latest"]
)

PRIOR_MONTH = (datetime.datetime.strptime(REPORT_MONTH, "%Y/%m").replace(day=1) - datetime.timedelta(days=1)).strftime("%Y/%m")

LOOKBACK_START = "2024-01-01"
LOOKBACK_MONTH = LOOKBACK_START[:7].replace("-", "/")

# month-over-month flag: both conditions must hold, so low-volume products don't trigger on small absolute moves
MOM_PCT_THRESHOLD = 0.10
MOM_ABS_THRESHOLD = 50

print(f"Report Month:   {REPORT_MONTH}")
print(f"Prior Month:    {PRIOR_MONTH}")
print(f"Main Account:   {MAIN_ACCOUNT}")
print(f"PORT Account:   {PORT_ACCOUNT}")
print(f"Lookback Start: {LOOKBACK_START}")

In [0]:
# Verify each account has both summary and detail data for the report month.
# Bloomberg files are downloaded as one bulk export mid-month; if only one account's file
# has been ingested, the other account shows up as a drop to zero and triggers false flags.
COMPLETENESS = {}
for account_id, account_name in ACCOUNTS.items():
    has_summary = ss_df.where((col("ACCOUNT_ID") == account_id) & (col("REPORT_YR_MONTH") == REPORT_MONTH)).limit(1).count() > 0
    has_detail = sd_billed.where((col("ACCOUNT_ID") == account_id) & (col("REPORT_YR_MONTH") == REPORT_MONTH)).limit(1).count() > 0
    missing = []
    if not has_summary:
        missing.append("summary")
    if not has_detail:
        missing.append("detail")
    status = "PASS" if not missing else "INCOMPLETE"
    COMPLETENESS[account_id] = {"name": account_name, "status": status, "missing": missing}
    if status == "PASS":
        print(f"  {account_name} ({account_id}): PASS")
    else:
        print(f"  WARNING: {account_name} ({account_id}): INCOMPLETE — missing {', '.join(missing)}")

In [0]:
# Shared metric tables used by the Executive Summary, Reconciliation, Month-over-Month and Job Attribution sections.
# Unit of billing = one (BILLABLE_PRODUCT, ID_BB_UNIQUE) pair per account per month.
_keys = ["ACCOUNT_ID", "REPORT_YR_MONTH", "BILLABLE_PRODUCT", "ID_BB_UNIQUE"]

# --- Billed securities from the summary report (= invoice basis) -------------------------------
billed_securities = (
    ss_df
    .groupBy(_keys)
    .agg(
        F.max(col("TOTAL_USAGE") > 0).alias("IS_ACTIVE"),
        F.sum("GROUP_AC").alias("ACCESS"),
    )
    .withColumn("STATUS", F.when(col("IS_ACTIVE"), "ACTIVE").otherwise("INACTIVE"))
    .drop("IS_ACTIVE")
)

# --- Billed requests from the detail report, with the job that made them ------------------------
# JOB:
#   F (FTP)  Bxx files -> group by process, e.g. "FTP B02 Security";
#            other FTP files (History*, HDUMOS*, BDL_*) -> "FTP Ad-hoc requests"
#   H (Web Services) -> one bucket
#   other (e.g. P)   OUTPUT_ID is a named job -> use as-is
NON_PROD_ENVS = ["UAT", "ACC", "TST", "TEST", "DEV"]

billed_requests = (
    sd_billed
    .withColumn(
        "JOB",
        F.when(
            (col("SERVICE") == "F") & col("OUTPUT_ID").rlike(r"^B\d\d_") & col("PROCESS_NAME").isNotNull(),
            F.concat_ws(" ", F.lit("FTP"), col("PROCESS_ID"), col("PROCESS_NAME")),
        )
        .when(col("SERVICE") == "H", F.lit("Web Services (per request)"))
        .when(
            (col("SERVICE") == "F") & ~col("OUTPUT_ID").rlike(r"^B\d\d_"),
            F.lit("FTP Ad-hoc requests"),
        )
        .otherwise(col("OUTPUT_ID")),
    )
    .select(*_keys, "JOB", "SERVICE", "ENV")
    .distinct()
)

# --- Summary vs detail reconciliation ------------------------------------------------------------
# INACTIVE billed securities (no usage this month, carried from earlier downloads) have no detail rows, so:
#   MATCH  <=>  every ACTIVE summary security is in the detail (summary_only_active = 0)
#          and  every billed detail security is in the summary (detail_only = 0)
_sd_keys = billed_requests.select(_keys).distinct().withColumn("IN_DETAIL", F.lit(True))

recon_df = (
    billed_securities.withColumn("IN_SUMMARY", F.lit(True))
    .join(_sd_keys, _keys, "full_outer")
    .groupBy("ACCOUNT_ID", "REPORT_YR_MONTH", "BILLABLE_PRODUCT")
    .agg(
        F.count(F.when(col("IN_SUMMARY"), 1)).alias("billed_total"),
        F.count(F.when(col("STATUS") == "ACTIVE", 1)).alias("billed_active"),
        F.count(F.when(col("STATUS") == "INACTIVE", 1)).alias("billed_inactive"),
        F.coalesce(F.sum("ACCESS"), F.lit(0)).alias("access"),
        F.count(F.when(col("IN_DETAIL"), 1)).alias("detail_billed"),
        F.count(F.when(col("IN_SUMMARY") & col("IN_DETAIL").isNull(), 1)).alias("billed_not_requested"),
        F.count(F.when((col("STATUS") == "ACTIVE") & col("IN_DETAIL").isNull(), 1)).alias("summary_only_active"),
        F.count(F.when(col("IN_SUMMARY").isNull() & col("IN_DETAIL"), 1)).alias("detail_only"),
    )
    .withColumn("inactive_pct", F.round(col("billed_inactive") / col("billed_total"), 3))
    .withColumn("status", F.when((col("summary_only_active") == 0) & (col("detail_only") == 0), "MATCH").otherwise("MISMATCH"))
)

# --- Month-over-month change per account and product (REPORT_MONTH vs PRIOR_MONTH) ---------------
_cur = recon_df.where(col("REPORT_YR_MONTH") == REPORT_MONTH).select("ACCOUNT_ID", "BILLABLE_PRODUCT", col("billed_total").alias("current"))
_pri = recon_df.where(col("REPORT_YR_MONTH") == PRIOR_MONTH).select("ACCOUNT_ID", "BILLABLE_PRODUCT", col("billed_total").alias("prior"))

mom_df = (
    _cur.join(_pri, ["ACCOUNT_ID", "BILLABLE_PRODUCT"], "full_outer")
    .fillna(0, subset=["current", "prior"])
    .withColumn("delta", col("current") - col("prior"))
    .withColumn("pct", F.when(col("prior") > 0, F.round(col("delta") / col("prior"), 3)))
    .withColumn(
        "flag",
        ((F.abs(col("pct")) >= MOM_PCT_THRESHOLD) | col("pct").isNull()) & (F.abs(col("delta")) >= MOM_ABS_THRESHOLD),
    )
)

# --- Securities added / dropped between PRIOR_MONTH and REPORT_MONTH, attributed to jobs ----------
_sec_cur = billed_securities.where(col("REPORT_YR_MONTH") == REPORT_MONTH).select("ACCOUNT_ID", "BILLABLE_PRODUCT", "ID_BB_UNIQUE")
_sec_pri = billed_securities.where(col("REPORT_YR_MONTH") == PRIOR_MONTH).select("ACCOUNT_ID", "BILLABLE_PRODUCT", "ID_BB_UNIQUE")

def _with_jobs(securities, month, unattributed_label):
    """Attach the job(s) that requested each security in `month`; securities with no billed request get a label."""
    jobs = billed_requests.where(col("REPORT_YR_MONTH") == month).select("ACCOUNT_ID", "BILLABLE_PRODUCT", "ID_BB_UNIQUE", "JOB")
    return (
        securities.join(jobs, ["ACCOUNT_ID", "BILLABLE_PRODUCT", "ID_BB_UNIQUE"], "left")
        .withColumn("JOB", F.coalesce(col("JOB"), F.lit(unattributed_label)))
    )

added_df = _with_jobs(_sec_cur.subtract(_sec_pri), REPORT_MONTH, "Billed without request this month").withColumn("CHANGE", F.lit("added"))
dropped_df = _with_jobs(_sec_pri.subtract(_sec_cur), PRIOR_MONTH, "Carried over, no longer billed").withColumn("CHANGE", F.lit("dropped"))
changes_df = added_df.unionByName(dropped_df)

print(f"Core metrics built for {REPORT_MONTH} vs {PRIOR_MONTH}")